In [1]:
import asyncio
import io
import json
import os

import aioboto3
import pandas as pd
from loguru import logger

In [2]:
EXECUTION_LEVEL = os.getenv("EXECUTION_LEVEL", "local") # if local then fall back to local execution

BUCKET_NAME = os.getenv("BUCKET_NAME", "rearc-quest")
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

if EXECUTION_LEVEL == "local":
    AWS_ENDPOINT_URL = os.getenv("AWS_ENDPOINT_URL", "http://localhost:9000")
    AWS_ACCESS_KEY_ID = os.getenv("AWS_ACCESS_KEY_ID", "minioadmin")
    AWS_SECRET_ACCESS_KEY = os.getenv("AWS_SECRET_ACCESS_KEY", "minioadmin")
else:
    AWS_ENDPOINT_URL = None
    AWS_ACCESS_KEY_ID = None
    AWS_SECRET_ACCESS_KEY = None

FILE_NAME_1 = os.getenv("FILE_NAME_1", "pr.data.0.Current")
FILE_NAME_2 = os.getenv("FILE_NAME_2", "step_2.json")
SERIES_ID = os.getenv("SERIES_ID", "PRS30006032")
QUATER = os.getenv("QUATER","Q01")

In [3]:
async def download_from_s3():

    session = aioboto3.Session(
        aws_access_key_id=AWS_ACCESS_KEY_ID,
        aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
        region_name=AWS_REGION,
    )
    file_1_buffer = io.BytesIO()
    file_2_buffer = io.BytesIO()
    async with session.client("s3", endpoint_url=AWS_ENDPOINT_URL) as s3_client:
        logger.info("Files downloading started")
        await s3_client.download_fileobj(
            Bucket=BUCKET_NAME, Key=FILE_NAME_1, Fileobj=file_1_buffer
        )
        await s3_client.download_fileobj(
            Bucket=BUCKET_NAME, Key=FILE_NAME_2, Fileobj=file_2_buffer
        )
        file_1_buffer.seek(0)
        file_2_buffer.seek(0)
        logger.info("Files downloading completed successfully")

    return file_1_buffer, file_2_buffer

In [4]:
async def download_and_convert_to_dataframe() -> tuple[pd.DataFrame, pd.DataFrame]:
    file_1_buffer, file_2_buffer = await download_from_s3()
    logger.info("Files downloaded successfully")
    df1 = pd.read_csv(file_1_buffer, sep="\t")
    # dataframe 1 needs column name renaming
    columns = df1.columns.tolist()
    df1.columns = [col.strip() for col in columns]
    # file 2 is little trickier
    df2 = pd.DataFrame(json.load(file_2_buffer).get("data", []))
    return df1, df2

In [5]:
df1, df2 = await download_and_convert_to_dataframe()

2026-03-12 06:55:36.335 | INFO     | __main__:download_from_s3:11 - Files downloading started
2026-03-12 06:55:36.385 | INFO     | __main__:download_from_s3:20 - Files downloading completed successfully
2026-03-12 06:55:36.387 | INFO     | __main__:download_and_convert_to_dataframe:3 - Files downloaded successfully


In [6]:
filtered = df2[(df2["Year"] >= 2013) & (df2["Year"] <= 2018)]

mean_population = filtered["Population"].mean()
std_population = filtered["Population"].std()

logger.info(f"Mean Population: {mean_population}")
logger.info(f"Standard Deviation: {std_population}")

2026-03-12 06:55:36.447 | INFO     | __main__:<module>:6 - Mean Population: 322069808.0
2026-03-12 06:55:36.449 | INFO     | __main__:<module>:7 - Standard Deviation: 4158441.040908095


In [7]:
# Step 1: yearly sum per series
yearly = df1.groupby(["series_id", "year"], as_index=False)["value"].sum()
# Step 2: find best year per series
result = yearly.loc[yearly.groupby("series_id")["value"].idxmax()]

In [8]:
result

,series_id,year,value
27,PRS30006011,2022,20.500
58,PRS30006012,2022,17.100
65,PRS30006013,1998,705.895
108,PRS30006021,2010,17.700
139,PRS30006022,2010,12.400
...,...,...,...
8459,PRS88003192,2002,282.800
8512,PRS88003193,2024,862.564
8541,PRS88003201,2022,38.900
8572,PRS88003202,2022,29.700


In [11]:
df1["period"] = df1["period"].str.strip()
df1["series_id"] = df1["series_id"].str.strip()
filtered = df1[(df1["series_id"] == SERIES_ID) & (df1["period"] == QUATER)]
filtered = filtered[["series_id", "period", "value", "year"]]

# step2 : select needed columns from df_2
df_2 = df2[["Population", "Year"]].rename(columns={"Year": "year"})

# Step 3: merge dataframes and filter the notna values
merged = pd.merge(filtered, df_2, on="year", how="left")
merged = merged[merged["Population"].notna()]

In [12]:
merged

,series_id,period,value,year,Population
18,PRS30006032,Q01,0.5,2013,316128839.0
19,PRS30006032,Q01,-0.1,2014,318857056.0
20,PRS30006032,Q01,-1.7,2015,321418821.0
21,PRS30006032,Q01,-1.4,2016,323127515.0
22,PRS30006032,Q01,0.9,2017,325719178.0
23,PRS30006032,Q01,0.5,2018,327167439.0
24,PRS30006032,Q01,-1.6,2019,328239523.0
26,PRS30006032,Q01,0.5,2021,331893745.0
27,PRS30006032,Q01,5.3,2022,333287562.0
28,PRS30006032,Q01,0.5,2023,334914896.0
